In [ ]:
# ============================================================
# INDUSTRIALISATION — recalage natif -> MNI pour TOUS les patients
# Reprend où il s'est arrêté, log les erreurs, QC automatique
# ============================================================

!pip install antspyx -q
import ants, numpy as np, os, glob, csv, traceback

base    = "/content/drive/MyDrive/isles_prp/MNI"
qc_dir  = f"{base}/_QC"          # images de contrôle qualité
log_csv = f"{base}/_recalage_log.csv"
os.makedirs(qc_dir, exist_ok=True)

mni = ants.image_read(ants.get_ants_data("mni"))

# Liste tous les dossiers patients
patients = sorted([d for d in os.listdir(base)
                   if d.startswith("sub-stroke")
                   and os.path.isdir(f"{base}/{d}")])
print(f"{len(patients)} patients détectés.")

# Reprise : charge les patients déjà traités avec succès
deja_fait = set()
if os.path.exists(log_csv):
    with open(log_csv) as f:
        for row in csv.DictReader(f):
            if row["statut"] == "OK":
                deja_fait.add(row["patient"])
print(f"{len(deja_fait)} déjà traités, seront ignorés.")

# Ouvre le log en mode append
fichier_existe = os.path.exists(log_csv)
log = open(log_csv, "a", newline="")
writer = csv.writer(log)
if not fichier_existe:
    writer.writerow(["patient","statut","vol_natif_cm3",
                     "vol_mni_cm3","ratio","voxels_mni","message"])

for i, pid in enumerate(patients):
    if pid in deja_fait:
        continue
    print(f"\n[{i+1}/{len(patients)}] {pid} ...", end=" ")
    try:
        dwi_p = f"{base}/{pid}/{pid}_ses-01_dwi.nii.gz"
        msk_p = f"{base}/{pid}/{pid}_ses-01_lesion-msk.nii.gz"
        if not (os.path.exists(dwi_p) and os.path.exists(msk_p)):
            raise FileNotFoundError("DWI ou masque manquant")

        dwi = ants.image_read(dwi_p)
        msk = ants.image_read(msk_p)

        reg = ants.registration(fixed=mni, moving=dwi, type_of_transform="SyN")
        msk_mni = ants.apply_transforms(
            fixed=mni, moving=msk,
            transformlist=reg["fwdtransforms"],
            interpolator="nearestNeighbor")

        # Sauvegarde
        out_dir = f"{base}/{pid}/registered"
        os.makedirs(out_dir, exist_ok=True)
        ants.image_write(reg["warpedmovout"], f"{out_dir}/{pid}_dwi_MNI.nii.gz")
        ants.image_write(msk_mni,             f"{out_dir}/{pid}_lesion_MNI.nii.gz")

        # Métriques
        vn = float(msk.sum())     * np.prod(msk.spacing)     / 1000.0
        vm = float(msk_mni.sum()) * np.prod(msk_mni.spacing) / 1000.0
        ratio = vm/vn if vn > 0 else 0

        # Image QC sauvegardée automatiquement
        ants.plot(mni, overlay=msk_mni, overlay_alpha=0.6, axis=2,
                  nslices=8, figsize=2,
                  filename=f"{qc_dir}/{pid}_QC.png")

        writer.writerow([pid,"OK",f"{vn:.2f}",f"{vm:.2f}",
                         f"{ratio:.2f}",int(msk_mni.sum()),""])
        log.flush()
        print(f"OK (ratio {ratio:.2f})")

    except Exception as e:
        writer.writerow([pid,"ERREUR","","","","",str(e)])
        log.flush()
        print(f"ERREUR : {e}")
        traceback.print_exc()

log.close()
print("\n=== Terminé. Log :", log_csv, "| QC :", qc_dir, "===")

In [ ]:
import pickle
import numpy as np

# Le fichier est à la racine de isles_prp, pas dans MNI
base = "/content/drive/MyDrive/isles_prp/MNI"
f = f"{base}/sub-stroke0189_lesion_MNI_nemo_output_ifod2act_chacoconn_cocoyeo143subj_mean.pkl"

# Charger
data = pickle.load(open(f, "rb"))

# chacoconn est sparse et triangulaire supérieure -> dense + symétriser
mat = data.toarray() if hasattr(data, "toarray") else np.array(data)
mat = mat + mat.T          # reconstruire la matrice complète symétrique

# --- Vérifications ---
print("="*50)
print("VALIDATION DE LA MATRICE DE DÉCONNEXION")
print("="*50)
print("Forme de la matrice :", mat.shape)
print("Valeurs non nulles  :", int(np.count_nonzero(mat)))
print("Valeur min / max    :", round(float(mat.min()), 4), "/", round(float(mat.max()), 4))
print("Nombre de régions touchées :", int(np.count_nonzero(mat.sum(axis=0))))
print("="*50)

# Sauvegarder en CSV
out_csv = f"{base}/chacoconn_0189.csv"
np.savetxt(out_csv, mat, delimiter=",")
print(f"Matrice sauvegardée : {out_csv}")

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Charger la matrice
base = "/content/drive/MyDrive/isles_prp/MNI"
f = f"{base}/sub-stroke0189_lesion_MNI_nemo_output_ifod2act_chacoconn_cocoyeo143subj_mean.pkl"
data = pickle.load(open(f, "rb"))
mat = data.toarray() if hasattr(data, "toarray") else np.array(data)
mat = np.maximum(mat, mat.T)  # Symétrise en prenant le max (corrige la surestimation du +)

# Heatmap
plt.figure(figsize=(12, 10))
plt.imshow(mat, cmap='hot', interpolation='nearest')
plt.colorbar(label='ChaCo (déconnexion)')
plt.title('Matrice de déconnexion région-à-région - Patient 0189')
plt.xlabel('Région')
plt.ylabel('Région')
plt.tight_layout()
plt.savefig(f"{base}/graphe_0189_heatmap.png", dpi=150)
print("Heatmap sauvegardée.")
plt.show()

In [ ]:
import pickle
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Charger
base = "/content/drive/MyDrive/isles_prp/MNI"
f = f"{base}/sub-stroke0189_lesion_MNI_nemo_output_ifod2act_chacoconn_cocoyeo143subj_mean.pkl"
data = pickle.load(open(f, "rb"))
mat = data.toarray() if hasattr(data, "toarray") else np.array(data)
mat = np.maximum(mat, mat.T)

# Ne garder que les connexions significatives (seuil pour visibilité)
threshold = 0.3  # ajuste selon ce que tu veux voir
mat_thresh = mat.copy()
mat_thresh[mat_thresh < threshold] = 0

# Créer le graphe NetworkX
G = nx.from_numpy_array(mat_thresh)

# Visualiser (attention : 143 nœuds = dense, peut être illisible)
plt.figure(figsize=(14, 14))
pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)
edges = G.edges()
weights = [G[u][v]['weight'] for u,v in edges]

nx.draw_networkx_nodes(G, pos, node_size=50, node_color='skyblue', alpha=0.8)
nx.draw_networkx_edges(G, pos, width=[w*2 for w in weights], alpha=0.4, edge_color='red')

plt.title(f'Réseau de déconnexion (seuil ChaCo > {threshold}) - Patient 0189')
plt.axis('off')
plt.tight_layout()
plt.savefig(f"{base}/graphe_0189_network.png", dpi=150)
print("Graphe réseau sauvegardé.")
plt.show()

In [ ]:
import pickle
import numpy as np
import networkx as nx

# Charger
base = "/content/drive/MyDrive/isles_prp/MNI"
f = f"{base}/sub-stroke0189_lesion_MNI_nemo_output_ifod2act_chacoconn_cocoyeo143subj_mean.pkl"
data = pickle.load(open(f, "rb"))
mat = data.toarray() if hasattr(data, "toarray") else np.array(data)
mat = np.maximum(mat, mat.T)

# Créer graphe NetworkX
G = nx.from_numpy_array(mat)

print("="*50)
print("STATISTIQUES DU GRAPHE")
print("="*50)
print(f"Nombre de nœuds : {G.number_of_nodes()}")
print(f"Nombre d'arêtes : {G.number_of_edges()}")
print(f"Densité du graphe : {nx.density(G):.3f}")
print(f"Degré moyen : {np.mean([d for n, d in G.degree()]):.1f}")
print(f"Poids moyen des arêtes : {np.mean([d['weight'] for u,v,d in G.edges(data=True)]):.3f}")
print("\nTop 10 régions par degré (les plus connectées) :")
degrees = sorted(G.degree(), key=lambda x: x[1], reverse=True)[:10]
for node, deg in degrees:
    print(f"  Région {node}: {deg} connexions")

In [ ]:
import shutil
import os

# Supprimer le contenu résiduel du point de montage
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive')
    print("Point de montage nettoyé.")

# Maintenant monter proprement
from google.colab import drive
drive.mount('/content/drive')
print("Drive monté.")

In [ ]:
import os, pandas as pd
base = "/content/drive/MyDrive/isles_prp"
# Tous les dossiers sub-stroke
tous = sorted([d for d in os.listdir(base) if d.startswith("sub-stroke")])
print(f"Total dossiers sub-stroke : {len(tous)}")
print(f"Premier : {tous[0]} | Dernier : {tous[-1]}")
# Vérifier le log
log = pd.read_csv(f"{base}/MNI/_recalage_log.csv")
print(f"\nDans le log : {len(log)} patients")
print(log["statut"].value_counts())

In [ ]:
import pandas as pd
base = "/content/drive/MyDrive/isles_prp/MNI"
df = pd.read_csv(f"{base}/_recalage_log.csv")
df_ok = df[df.statut == "OK"].copy()
df_ok["ratio"] = pd.to_numeric(df_ok["ratio"], errors="coerce")

print("=== DISTRIBUTION DES RATIOS DE VOLUME ===")
print(df_ok["ratio"].describe())
print()
print(f"Ratio 0.7-1.3 (bon)        : {((df_ok.ratio >= 0.7) & (df_ok.ratio <= 1.3)).sum()} / {len(df_ok)}")
print(f"Ratio > 1.3 (gonflement)   : {(df_ok.ratio > 1.3).sum()}")
print(f"Ratio > 1.5 (gros gonfl.)  : {(df_ok.ratio > 1.5).sum()}")
print(f"Ratio < 0.7 (perte)        : {(df_ok.ratio < 0.7).sum()}")
print()
print("=== Top 10 ratios les plus élevés (à vérifier visuellement) ===")
print(df_ok.nlargest(10, "ratio")[["patient", "vol_natif_cm3", "vol_mni_cm3", "ratio"]].to_string(index=False))
print()
print("=== Patients en ERREUR ===")
print(df[df.statut == "ERREUR"][["patient", "message"]].to_string(index=False))

In [ ]:
!pip install antspyx -q

In [ ]:
import ants, numpy as np


base = "/content/drive/MyDrive/isles_prp/MNI"
pid = "sub-stroke0143"

dwi = ants.image_read(f"{base}/{pid}/{pid}_ses-01_dwi.nii.gz")
msk = ants.image_read(f"{base}/{pid}/{pid}_ses-01_lesion-msk.nii.gz")
mni = ants.image_read(ants.get_ants_data("mni"))

# NOUVEAU : pré-resampling isotrope avant recalage
dwi_iso = ants.resample_image(dwi, (1.0, 1.0, 1.0), use_voxels=False, interp_type=4)  # linéaire
msk_iso = ants.resample_image(msk, (1.0, 1.0, 1.0), use_voxels=False, interp_type=1)  # NN pour masque

print("DWI natif :", dwi.shape, "voxels :", dwi.spacing)
print("DWI isotrope :", dwi_iso.shape, "voxels :", dwi_iso.spacing)

# Recalage à partir de l'image isotrope
reg = ants.registration(fixed=mni, moving=dwi_iso, type_of_transform="SyN")
msk_mni = ants.apply_transforms(fixed=mni, moving=msk_iso,
                                 transformlist=reg["fwdtransforms"],
                                 interpolator="nearestNeighbor")

vn = float(msk.sum()) * np.prod(msk.spacing) / 1000
vm = float(msk_mni.sum()) * np.prod(msk_mni.spacing) / 1000
print(f"\nVolume natif : {vn:.2f} cm³")
print(f"Volume MNI   : {vm:.2f} cm³")
print(f"Ratio        : {vm/vn:.2f}")

In [ ]:
import ants
base = "/content/drive/MyDrive/isles_prp/MNI"
pid = "sub-stroke0143"

dwi_mni = ants.image_read(f"{base}/{pid}/registered/{pid}_dwi_MNI.nii.gz")
msk_mni = ants.image_read(f"{base}/{pid}/registered/{pid}_lesion_MNI.nii.gz")
mni = ants.image_read(ants.get_ants_data("mni"))

# Le DWI recalé est-il bien aligné sur le MNI ?
print("=== DWI recalé sur MNI (vérifier l'alignement cérébral) ===")
ants.plot(mni, overlay=dwi_mni, overlay_alpha=0.4, axis=2, nslices=8, figsize=3)

# Le masque tombe-t-il dans le cerveau et à un endroit cohérent ?
print("\n=== Masque sur MNI ===")
ants.plot(mni, overlay=msk_mni, overlay_alpha=0.6, axis=2, nslices=8, figsize=3)

In [ ]:
import nibabel as nib
import numpy as np
msk = nib.load(f"{base}/{pid}/{pid}_ses-01_lesion-msk.nii.gz").get_fdata()
print(f"Valeurs uniques dans le masque : {np.unique(msk)}")
print(f"Voxels non nuls : {int((msk != 0).sum())}")
print(f"Min : {msk.min()}, Max : {msk.max()}")

In [ ]:
import nibabel as nib
msk_nat = nib.load(f"{base}/{pid}/{pid}_ses-01_lesion-msk.nii.gz")
msk_mni_img = nib.load(f"{base}/{pid}/registered/{pid}_lesion_MNI.nii.gz")

print("=== NATIF ===")
print(f"Shape : {msk_nat.shape}")
print(f"Spacing : {msk_nat.header.get_zooms()}")
print(f"Voxels non nuls : {int((msk_nat.get_fdata() != 0).sum())}")

print("\n=== MNI ===")
print(f"Shape : {msk_mni_img.shape}")
print(f"Spacing : {msk_mni_img.header.get_zooms()}")
print(f"Voxels non nuls : {int((msk_mni_img.get_fdata() != 0).sum())}")

In [ ]:
!pip install nilearn -q

In [ ]:
import pandas as pd
from nilearn import plotting, datasets

base = "/content/drive/MyDrive/isles_prp/MNI"

df = pd.read_csv(f"{base}/_recalage_log.csv")
df_ok = df[df.statut == "OK"].copy()
df_ok["ratio"] = pd.to_numeric(df_ok["ratio"], errors="coerce")

# Échantillon stratifié par ratio
low  = df_ok[df_ok.ratio.between(0.9, 1.1)].sample(5, random_state=42)
mid  = df_ok[df_ok.ratio.between(1.3, 1.5)].sample(5, random_state=42)
high = df_ok[df_ok.ratio > 1.5].sample(5, random_state=42)
sample = pd.concat([low, mid, high])

# Template MNI de nilearn
mni_template = datasets.load_mni152_template(resolution=1)

for _, row in sample.iterrows():
    pid = row['patient']
    msk_path = f"{base}/{pid}/registered/{pid}_lesion_MNI.nii.gz"
    print(f"\n{pid} - ratio {row['ratio']:.2f}")
    display(plotting.view_img(msk_path, bg_img=mni_template,
                              threshold=0.5, opacity=0.6,
                              title=f"{pid} (ratio={row['ratio']:.2f})"))

In [ ]:
from nilearn import plotting, datasets

base = "/content/drive/MyDrive/isles_prp/MNI"
pid = "sub-stroke0143"

# Natif : masque sur DWI natif
print("=== NATIF ===")
display(plotting.view_img(
    f"{base}/{pid}/{pid}_ses-01_lesion-msk.nii.gz",
    bg_img=f"{base}/{pid}/{pid}_ses-01_dwi.nii.gz",
    threshold=0.5, opacity=0.6,
    title=f"{pid} - NATIF"
))

# MNI : masque recalé sur template MNI
print("=== MNI ===")
display(plotting.view_img(
    f"{base}/{pid}/registered/{pid}_lesion_MNI.nii.gz",
    bg_img=datasets.load_mni152_template(resolution=1),
    threshold=0.5, opacity=0.6,
    title=f"{pid} - MNI"
))

In [ ]:
import ants
mni_path = ants.get_ants_data("mni")
print("Chemin réel :", mni_path)

In [ ]:
import pandas as pd
import os

base = "/content/drive/MyDrive/isles_prp/MNI"
pid = "sub-stroke0133"

for csv_name in ["demographic_baseline", "outcome"]:
    f = f"{base}/{pid}/{pid}_ses-01_{csv_name}.csv"
    print(f"\n{csv_name}.csv :")
    print(f"  Taille : {os.path.getsize(f)} octets")
    if os.path.getsize(f) > 0:
        df = pd.read_csv(f)
        print(f"  Contenu : {df.shape}")
        print(df.head())
    else:
        print("  ⚠️  Fichier vide")

In [ ]:
import pandas as pd
import os

base = "/content/drive/MyDrive/isles_prp/MNI"

# Liste des patients OK depuis le log de recalage
df_log = pd.read_csv(f"{base}/_recalage_log.csv")
patients_ok = df_log[df_log.statut == "OK"]["patient"].unique().tolist()
print(f"Patients OK : {len(patients_ok)}")

# Charger TOUS les CSV démographique + outcome
all_data = []
for pid in patients_ok:
    demo_path = f"{base}/{pid}/{pid}_ses-01_demographic_baseline.csv"
    out_path  = f"{base}/{pid}/{pid}_ses-01_outcome.csv"

    if os.path.exists(demo_path) and os.path.getsize(demo_path) > 0:
        demo = pd.read_csv(demo_path)
        out = pd.read_csv(out_path) if os.path.exists(out_path) and os.path.getsize(out_path) > 0 else pd.DataFrame()
        row = pd.concat([demo, out], axis=1)
        row["patient"] = pid
        all_data.append(row)

cohort = pd.concat(all_data, ignore_index=True)
print(f"\nTaille du dataset consolidé : {cohort.shape}")
print(f"Colonnes : {list(cohort.columns)}")

# Taux de complétude par variable
print("\n=== TAUX DE COMPLÉTUDE PAR VARIABLE ===")
completude = (1 - cohort.isnull().mean()) * 100
print(completude.sort_values(ascending=False).to_string())

# Vérifier le label mRS à 3 mois
print(f"\n=== LABEL mRS 3 months ===")
if "mRS 3 months" in cohort.columns:
    label = cohort["mRS 3 months"]
    print(f"Non-NaN : {label.notna().sum()} / {len(label)}")
    print(f"Distribution :")
    print(label.value_counts().sort_index())
    print(f"\nDichotomie ≤2 (favorable) vs >2 (défavorable) :")
    binary = (label > 2).astype(int)
    print(f"  Favorable (≤2) : {(binary == 0).sum()}")
    print(f"  Défavorable (>2) : {(binary == 1).sum()}")
    print(f"  NaN : {label.isna().sum()}")

# Sauvegarder le dataset consolidé
cohort.to_csv(f"{base}/cohort_consolidated.csv", index=False)
print(f"\n✓ Dataset consolidé sauvegardé : cohort_consolidated.csv")

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/isles_prp/MNI/cohort_consolidated.csv")
df_unique = df.drop_duplicates("patient")
print(f"Avant : {len(df)} → Après dédup : {len(df_unique)}")

# Filtrer les patients avec label valide
df_final = df_unique.dropna(subset=["mRS 3 months"])
print(f"Avec label valide : {len(df_final)}")

# Ajouter la colonne binaire pour ton GNN
df_final["mRS_binary"] = (df_final["mRS 3 months"] > 2).astype(int)
df_final.to_csv("/content/drive/MyDrive/isles_prp/MNI/cohort_final.csv", index=False)
print(f"\n=== COHORTE FINALE ===")
print(f"N = {len(df_final)}")
print(f"Favorable (mRS≤2) : {(df_final.mRS_binary == 0).sum()}")
print(f"Défavorable (mRS>2) : {(df_final.mRS_binary == 1).sum()}")

In [ ]:
import os
import pandas as pd
import zipfile

# === Chemins ===
base = "/content/drive/MyDrive/isles_prp/MNI"
sortie_zips = f"{base}/zips_nemo"
os.makedirs(sortie_zips, exist_ok=True)

# === 1. Charger la cohorte finale (129 patients avec label) ===
cohort = pd.read_csv(f"{base}/cohort_final.csv")
patients = cohort["patient"].tolist()
print(f"Cohorte finale : {len(patients)} patients")

# === 2. Vérifier que tous les masques MNI existent avant de zipper ===
manquants = []
masques = []
for pid in patients:
    msk_path = f"{base}/{pid}/registered/{pid}_lesion_MNI.nii.gz"
    if os.path.exists(msk_path):
        masques.append((pid, msk_path))
    else:
        manquants.append(pid)

if manquants:
    print(f"\n⚠️  {len(manquants)} masques manquants :")
    for p in manquants:
        print(f"   - {p}")
    print("⚠️  Ces patients ne seront pas inclus dans les zips. À vérifier.")

print(f"\nMasques MNI prêts à zipper : {len(masques)}")

# === 3. Générer les zips par paquets de 10 ===
TAILLE_BATCH = 10
nb_zips = (len(masques) + TAILLE_BATCH - 1) // TAILLE_BATCH  # arrondi sup
print(f"Nombre de zips à créer : {nb_zips}")

for i in range(0, len(masques), TAILLE_BATCH):
    paquet = masques[i:i + TAILLE_BATCH]
    num_zip = i // TAILLE_BATCH + 1
    nom_zip = f"{sortie_zips}/batch_{num_zip:02d}.zip"

    with zipfile.ZipFile(nom_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for pid, chemin in paquet:
            # arcname = nom du fichier seul, sans le chemin complet
            zf.write(chemin, arcname=os.path.basename(chemin))

    print(f"  ✓ batch_{num_zip:02d}.zip créé ({len(paquet)} masques)")

# === 4. Vérification finale ===
print(f"\n=== RÉCAPITULATIF ===")
print(f"Dossier de sortie : {sortie_zips}")
zips_crees = sorted([f for f in os.listdir(sortie_zips) if f.endswith('.zip')])
print(f"Zips créés : {len(zips_crees)}")
for z in zips_crees:
    chemin_z = f"{sortie_zips}/{z}"
    taille_mo = os.path.getsize(chemin_z) / (1024*1024)
    # Compter les fichiers dans le zip
    with zipfile.ZipFile(chemin_z, 'r') as zf:
        nb_fichiers = len(zf.namelist())
    print(f"  {z} : {nb_fichiers} masques, {taille_mo:.1f} Mo")

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from nilearn import plotting, datasets, image
from scipy import ndimage

# === Paramètres ===
base = "/content/drive/MyDrive/isles_prp/MNI"
pid  = "sub-stroke0189"

dwi_nat = f"{base}/{pid}/{pid}_ses-01_dwi.nii.gz"
msk_nat = f"{base}/{pid}/{pid}_ses-01_lesion-msk.nii.gz"
dwi_mni = f"{base}/{pid}/registered/{pid}_dwi_MNI.nii.gz"
msk_mni = f"{base}/{pid}/registered/{pid}_lesion_MNI.nii.gz"

mni_template = datasets.load_mni152_template(resolution=1)

# ============================================================
# Helpers
# ============================================================
def best_axial_cuts(mask_path, n=7):
    """Retourne n coords z (mm) sur les coupes les plus lésionnelles."""
    m = nib.load(mask_path)
    data = m.get_fdata() > 0
    per_slice = data.sum(axis=(0, 1))          # voxels lésionnels par coupe z
    if per_slice.sum() == 0:
        return None
    k_idx = np.argsort(per_slice)[::-1]
    k_idx = k_idx[per_slice[k_idx] > 0][:n]    # garde les coupes non vides
    k_idx = np.sort(k_idx)                      # ordre anatomique
    # indices voxel -> coords mm via l'affine
    aff = m.affine
    z_mm = [nib.affines.apply_affine(aff, [data.shape[0]//2,
                                           data.shape[1]//2, int(k)])[2]
            for k in k_idx]
    return z_mm

def lesion_stats(mask_path):
    m = nib.load(mask_path)
    data = m.get_fdata() > 0
    nvox = int(data.sum())
    vox_vol_ml = np.abs(np.linalg.det(m.affine[:3, :3])) / 1000.0
    return nvox, nvox * vox_vol_ml

def robust_window(img_path):
    d = nib.load(img_path).get_fdata()
    d = d[d > 0]
    return np.percentile(d, 2), np.percentile(d, 98)

# Stats lésion
nvox_nat, vol_nat = lesion_stats(msk_nat)
nvox_mni, vol_mni = lesion_stats(msk_mni)

# Coupes : on aligne les deux espaces sur les coupes MNI les plus lésionnelles
cuts_mni = best_axial_cuts(msk_mni, n=7)
cuts_nat = best_axial_cuts(msk_nat, n=7)

vmin_nat, vmax_nat = robust_window(dwi_nat)
vmin_mni, vmax_mni = robust_window(dwi_mni)

# ============================================================
# FIGURE 1 — Comparaison natif vs MNI (contours lésion)
# ============================================================
fig, axes = plt.subplots(2, 1, figsize=(16, 7), facecolor='white')

# --- NATIF ---
disp = plotting.plot_anat(
    dwi_nat, display_mode='z', cut_coords=cuts_nat,
    axes=axes[0], annotate=False, draw_cross=False, black_bg=False,
    vmin=vmin_nat, vmax=vmax_nat,
    title=(f"ESPACE NATIF — {pid} (DWI 0.41×0.41×2 mm)  |  "
           f"lésion : {nvox_nat} vox, {vol_nat:.1f} ml")
)
disp.add_contours(msk_nat, colors='red', linewidths=1.2, levels=[0.5])

# --- MNI ---
disp2 = plotting.plot_anat(
    mni_template, display_mode='z', cut_coords=cuts_mni,
    axes=axes[1], annotate=False, draw_cross=False, black_bg=False,
    title=(f"ESPACE MNI — {pid} (template 182×218×182, 1 mm iso)  |  "
           f"lésion : {nvox_mni} vox, {vol_mni:.1f} ml")
)
disp2.add_contours(msk_mni, colors='red', linewidths=1.2, levels=[0.5])

ratio = vol_mni / vol_nat if vol_nat else float('nan')
plt.suptitle(f"Recalage natif → MNI : {pid} (ratio volume = {ratio:.2f})",
             fontsize=15, fontweight='bold', y=1.02)

out1 = f"{base}/comparaison_natif_mni_{pid}_v3.png"
plt.savefig(out1, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Figure comparaison : {out1}")

# ============================================================
# FIGURE 2 — QC Registration
# ============================================================
# Centre de masse lésion (coords MNI) pour le crosshair
m_mni = nib.load(msk_mni)
com_vox = ndimage.center_of_mass(m_mni.get_fdata() > 0)
com_mm  = nib.affines.apply_affine(m_mni.affine, com_vox)

# Resample DWI MNI sur la grille exacte du template (sécurité checkerboard)
dwi_mni_rs = image.resample_to_img(dwi_mni, mni_template,
                                   interpolation='continuous',
                                   force_resample=True)

fig2, axes2 = plt.subplots(3, 1, figsize=(16, 10), facecolor='white')

# (a) Checkerboard patient recalé ⊗ template
chk = image.math_img("a*0 + b", a=mni_template, b=dwi_mni_rs)  # placeholder grid base
d_tpl = mni_template.get_fdata()
d_pat = dwi_mni_rs.get_fdata()
# damier 16x16 voxels
gx, gy, gz = np.indices(d_tpl.shape)
board = (((gx // 16) + (gy // 16) + (gz // 16)) % 2).astype(bool)
# normalisation commune pour mélange visuel cohérent
def norm(x):
    lo, hi = np.percentile(x[x > 0], (2, 98));
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)
mix = np.where(board, norm(d_tpl), norm(d_pat))
chk_img = nib.Nifti1Image(mix, mni_template.affine)

disp_a = plotting.plot_anat(
    chk_img, display_mode='z', cut_coords=cuts_mni,
    axes=axes2[0], annotate=False, draw_cross=False, black_bg=False,
    title="QC (a) Checkerboard : template ⊗ DWI recalé (alignement aux interfaces)"
)

# (b) Contours du template sur le DWI recalé
disp_b = plotting.plot_anat(
    dwi_mni_rs, display_mode='z', cut_coords=cuts_mni,
    axes=axes2[1], annotate=False, draw_cross=False, black_bg=False,
    vmin=vmin_mni, vmax=vmax_mni,
    title="QC (b) Contours template MNI (jaune) sur DWI recalé"
)
disp_b.add_contours(mni_template, colors='yellow', linewidths=0.8,
                    levels=[np.percentile(d_tpl[d_tpl > 0], 50)])

# (c) Crosshair au centre de masse lésionnel
disp_c = plotting.plot_roi(
    msk_mni, bg_img=mni_template, display_mode='ortho',
    cut_coords=com_mm, axes=axes2[2], annotate=True, draw_cross=True,
    black_bg=False, cmap='autumn', alpha=0.7,
    title=(f"QC (c) Centre de masse lésion @ MNI "
           f"[{com_mm[0]:.0f}, {com_mm[1]:.0f}, {com_mm[2]:.0f}] mm")
)

out2 = f"{base}/qc_registration_{pid}.png"
plt.savefig(out2, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f"✓ Figure QC : {out2}")